# Code : Calcul d'endomorphisme

In [1]:
#Evalution d'endomorphisme : Problème de division par fm

from sage.schemes.elliptic_curves.hom_frobenius import EllipticCurveHom_frobenius

def base_fwk(b,dk,f):
    
    #Dans K = Q(rk) quadratique de discriminant dk, on se donne b = x + y*rk dans un ordre O
    #On veut écrire b dans la base [1,f*wk] de l'ordre O, wk = (dk + rk)/2.
    
    x = b[0]
    y = b[1]
    xw = x - dk*y
    yw = (2*y)/f   #xw, yw sont dans ZZ. 
    return xw,yw
    
def base_fq1(b,fm,dk,f,tracef):
    
    # On représente le froebenius fq par le complexe (tracef + fm*rk)/2 dans K = Q(rk).
    # Remarque : il est a priori possible que fq soit représenté par le conjugué (tracef - fm*rk)/2 .

    # On écrit b = (xf + yf*fq)/fm, et on calcule xf, yf dans ZZ.
    
    # Il existe s1 entier tel que fq = fm*wk + s1.
    
    
    s1 = (-fm*dk + tracef)/2
    (x,y) = base_fwk(b,dk,f)
    xf = fm*x - y*s1*f
    yf = y*f
    return xf,yf


def eval_fq(P,kq,E):

    #calcul Frobenius_q(P) sur une courbe E définie sur Fq, avec q = p^(kq) et p premier.
    #On suppose que P appartient à E(Fq).
    
    if P == E(0):
        return P
    else :
        frob = EllipticCurveHom_frobenius(E, kq)
        return frob(P)

def eval_theta(P,E,K,O,b,c):

    #On écrit theta = (xf + yf*fp)/fm, où fp est le frobenius et fm son conducteur. Puis on calcul theta(P).
    #On suppose E définie sur Fp
    #Condition de calcul : fm premier avec le cardinal de E.
    
    theta = ((b*(c.conjugate())).gens_reduced())[0] 
    assert theta in O
    rK = K.gens()[0]
    dK = K.discriminant()
    f = O.conductor()
    D = (f^2)*dK
    rD = f*rK
    wK = (dK + rK)/2
    p = E.base_ring().characteristic()
    tracef = E.trace_of_frobenius()
    CE = 1 + p - tracef
    Dm = tracef^2 - 4*p
    fm = sqrt(Dm/dK)
    if gcd(fm,CE) == 1:
        inv_fm = Mod(fm,CE)^(-1)
    else:
        raise ValueError("Le conducteur du frobenius n'est pas premier avec le cardinal de E")
        
    xf, yf = base_fq1(theta,fm,dK,f,tracef)
    xfP = xf*P
    yfP = yf*P
    FyfP = eval_fq(yfP,1,E) #Suppose que E est définie sur Fp
    imP = inv_fm*(xfP + FyfP)
    
    return imP

In [2]:
# Alternative : Avec Squarevélu, pas besoin de division par fm

from sage.rings.finite_rings.integer_mod import square_root_mod_prime
from sage.rings.finite_rings.integer_mod import square_root_mod_prime_power
from sage.rings.number_field.order_ideal import NumberFieldOrderIdeal
from sage.schemes.elliptic_curves.hom_velusqrt import EllipticCurveHom_velusqrt
from sage.schemes.elliptic_curves.weierstrass_morphism import *
from sage.groups.generic import order_from_multiple
from sage.schemes.elliptic_curves.ell_curve_isogeny import compute_isogeny_bmss
from sage.schemes.elliptic_curves.hom_frobenius import EllipticCurveHom_frobenius
from sage.misc.search import search
import time

def ideal_de_norme(l,f,D):

    #Trouver un idéal de norme l premier.
    #On se place dans un ordre de discriminant D et de conducteur f.
    
    if kronecker(D,l) == 1:
        D = Mod(D,l)
        sq = square_root_mod_prime(D,l)
        sm = Mod(-sq,l)
        sq = ZZ(sq)
        sm = ZZ(sm)
        sq = min(sm,sq)
        #print(sq)
        return NumberFieldOrderIdeal(O,[l, -sq + f*rK]) 
    elif kronecker(D,l) == 0:
        return NumberFieldOrderIdeal(O,[l, f*rK])
    else:
        raise "l est inerte"

def forme_de_norme(l,D):

    #Calcule une forme quadratique binaire primitive normale de coefficient dominant l, de discriminant D. 
    #On demande à ce que l soit décomposé dans l'ordre de discriminant D.
    
    if kronecker(D,l) == 1:
        if l == 2 :
            d = Mod(D,8)
            b = square_root_mod_prime_power(d,2,3)
            b = ZZ(b)
        else :
            x = Mod(D,4)
            d = Mod(D,l)
            y = square_root_mod_prime(d,l)
            b = x.crt(y)
            b = ZZ(b)
        ql = BinaryQF([l,b, int(int((b^2 - D))/int(4*l))])
        return ql
    else:
        raise ValueError('l pas split')

def coeff_courbe_l_isogene(E,l,h,Psi_l):

    #Formules donnant un modèle rationel de la courbe image de la l-isogénie partant de E, de j-invariant h
    #Le polynôme modulaire Psi_l est précalculé. 
    
    F = E.base_field()
    j = E.j_invariant()
    a = E.a4()
    b = E.a6()
    
    ZXY = Psi_l.parent()
    (X,Y) = ZXY.gens()
    PsiX = (Psi_l.derivative(X))
    PsiX_eval = PsiX(j,h)
    PsiY = (Psi_l.derivative(Y))
    PsiY_eval = PsiY(j,h)
    assert PsiY_eval != 0
    
    h1 = F(-18/l)*(b/a)*(PsiX_eval/PsiY_eval)*j
    A = F(-1/48)*(h1^2)/((h-1728)*h)
    B = F(-1/864)*(h1^3)/((h-1728)*h^2)
    A = A*(F(l)^4) #Normalisation
    B = B*(F(l)^6)
    return A,B


def CheckElkies(E, ell, kernel_polynomial, lam):   #Implémentation de Pegasis, prise dans le fichier Elkies.py.
    r"""Given a kernel polynomial, verify it corresponds to the correct eigenvalue

    If the multiplication-by-lambda map has the following standard form in
    rational maps (c.f. Sutherland's lectures)

        [\lambda] = (u(x)/v(x), r(x, y)/s(x))

    then the eigenvalue is correct, if

        \pi(P) = \lambda P

    on all points in the kernel of the isogeny defined by kernel_polynomial.

    Note that r(x, y) = r(x, 1) * y, by the standard form of isogenies. So, if
    P = (x, y), this is equivalent to

        (x^p, y^p) = (u(x)/v(x), r(x, 1)/s(x) * y)

    for points in ker(\varphi)

    Verifying the first component is easy. To verify the second, we note that

            y^p = r(x, 1)/s(x) * y
        <=> y^{p-1} = r(x, 1)/s(x)
        <=> f(x)^{(p-1)/2} = r(x, 1)/s(x)
        <=> f(x)^{(p-1)/2} * s(x) = r(x, 1)

    where f(x) is the defining equation of the curve E: y^2 = f(x).
    """

    p = E.base_field().characteristic()
    E = E.short_weierstrass_model()

    # Defining equation of E: y^2 + g(x)y = f(x)
    f, g = E.hyperelliptic_polynomials()

    # Must be true, because E is in Weierstrass form
    assert g == 0

    # For efficiency: replace lambda with -lambda if -lambda has smaller
    # absolute value
    # If we switch, then we need to multiply the isogeny with -1
    # (which is multiplication by -1 on the y-coordinate)

    if lam > ell - lam:
        lam = ell - lam
        sign = -1
    else:
        sign = 1

    if lam == 1:
        Y = pow(f, (p - 1) / 2, kernel_polynomial)
        return sign * Y == 1

    # Build extension over which the x-coordinates of the kernel are defined
    extension = kernel_polynomial.parent().quotient_ring(kernel_polynomial)

    # Get rational functions of multiplication-by-lambda
    # x = u/v, y = r/x as in the description in the docstring
    x, y = E.multiplication_by_m(lam)[:2]
    u = extension(x.numerator())
    v = extension(x.denominator())
    # Overwrite r(x, y) with r(x, 1)
    r = y.numerator()
    r = extension(r(r.variables()[0], 1).univariate_polynomial())
    s = extension(y.denominator())

    # x^p in the extension
    Xp = extension(pow(kernel_polynomial.parent().gens()[0], p, kernel_polynomial))

    # Verify u(x)/v(x) = x^p
    if u != Xp * v:
        return False

    Y = extension(pow(f, (p - 1) / 2, kernel_polynomial))

    # Verify y^{p-1} = f(x)^{(p-1)/2} = sign * r(x, 1)/s(x)
    return Y * s == sign * r



def polynome_ker(E,l,L,f,fm,dk,tracef,j,j_prec,Psi_l):  #Algorithme 3 du rapport.

    #Calcul du polynôme décrivant le noyau de la l-isogénie définie par l'action de L sur E. 
    #Si on enchaine les calculs pour un même idéal L, j_prec est le j_invariants du dépard précédent. 
    #On suppose le polynôme modulaire Psi_l précalculé. 

    assert gcd(fm,l) == 1   #assure qu'il existe exactement deux l-isogénies rationnelles. 

    K.<t> = QuadraticField(dk) 
    Fq = E.base_ring()
    ql = L.quadratic_form()
    cible = (-ql[1] + f*t)/2   # Forme standard de L              
    (c,d) = base_fq1(cible, fm, dk, f, tracef)   # En particulier d = f est inversible modulo l par hypothèse.
    val = Mod(-c,l)*(Mod(d,l)^(-1))   # Valeur propre du frobenius sur le groupe de L-torsion.  
    val = ZZ(val)
    if abs(val) > abs(val - l):   # La valeur propre est définie modulo l. 
        val = val - l
    
    PFq.<z> = PolynomialRing(Fq)
    Psi = Psi_l(j,z)
    
    print('Recherche de j_invariants')
    racines = Psi.roots()
    
    h = racines[0][0]
    
    if j_prec != None:  #On déduit du calcul précédent que l'un des deux j_invariant correspond au dual/demi-tour.
        assert len(racines) == 2 or racines[0][1] == 2, f"racines was {racines}"
        if h == j_prec and len(racines) == 2:
            h = racines[1][0]
            
    print('j_invariants trouvés')
    assert h.parent() == E.base_ring()
    
    (A,B) = coeff_courbe_l_isogene(E,l,h,Psi_l)
    E_coef = EllipticCurve(Fq,[A,B])
    print('Choix j_invariant, application BMSS')
    Fker = compute_isogeny_bmss(E, E_coef, l) #On suppose p > 4l + 4 pour appliquer BMSS. 

    print('Verif choix j_invariant')
    if CheckElkies(E, l, Fker, val):
        print('bon j_invariant')
        return Fker
        
    else:  #Peut arriver s'il n'y a pas eu de calculs précédents. 
        print('changement de j_invariant')
        h = racines[1][0]
        (A,B)=coeff_courbe_l_isogene(E,l,h,Psi_l)
        E_coef = EllipticCurve(Fq,[A,B])
        print('Application BMSS')
        Fker = compute_isogeny_bmss(E, E_coef, l)
        if CheckElkies(E, l, Fker, val):
            return Fker
        else:
            raise ValueError('Aucun j_invariant correct') 

def phi_from_L_polynome(E,L,l,q,kq,f,fm,dk,tracef,j,j_prec,Psi_l):

    #Résume ce qui précéde pour déduire d'un idéal de norme l à une isogénie de degré l.
    
    print('Calcul du polynome du noyau')
    F_ker = polynome_ker(E,l,L,f,fm,dk,tracef,j,j_prec,Psi_l)
    print('Calcul isogénie par Vélu')
    phi = E.isogeny(F_ker) 
    return phi

def card_extension_noyau(E,l,L,f,fm,dk,tracef):       
    
    #On calcule l'extension Fker du corps de base, dans laquelle chercher le noyau de la l-isogénie définie par l'idéal L. 
    
    print('début calculs extensions noyaux')
    
    #On écrit L dans la base [1,fq]
    assert l == L.norm()
    assert is_prime(l), f" norme pas premier : {l}"

    K.<t> = QuadraticField(dk) 
    Fq = E.base_ring()
    q = Fq.cardinality()
    
    ql = L.quadratic_form()
    cible = (-ql[1] + f*t)/2            #Deuxieme élément d'une base sous forme standard .
    (c,d) = base_fq1(cible, fm, dk,f, tracef)     

    #On calcule le degré de l'extension : il ne dépend que de c et l (CF Miret Moreno prop2 + l-volcan kohel)

    if (fm//f)%l == 0:
        nb_iso = l+1
    else:
        if f%l != 0:
            nb_iso = kronecker(D,l) + 1 # ici nb_iso = 1 ou 2; 0 pas autorisé car L existe
        else:
            nb_iso = 1 #unique ascendante, ne devrais pas arriver
    
    print('f:',f,'fm :',fm)
    print('nb_iso:',nb_iso)
    
    if nb_iso == 2:
        Fl.<xl> = GF(l)
        b = GF(l)(-c)
        r = order_from_multiple(b, l - 1, operation='*')
        q_ker = q^r
        
    else:
        b = GF(l)(q)
        r = order_from_multiple(b, l - 1, operation='*')
        q_ker = q^r
        

    #On calcule le cardinal de la courbe E(Fker)
    print('debut calcul C_ker')
    Fker.<xk>, emb = Fq.extension(r, map = True)
    assert Fker.cardinality() == q_ker
    E_ker = EllipticCurve(Fker, [emb(E.a4()),emb(E.a6())])   #TODO Tester si corps de base non premier
    C_ker = E_ker.cardinality()    
    if C_ker%l != 0 :              #Par construction C_ker doit être divisible par l 
        r = 2*r
        q_ker = q_ker*q_ker
        Fker.<xk>, emb = Fq.extension(r, map = True)
        E_ker = EllipticCurve(Fker, [emb(E.a4()),emb(E.a6())]) 
        C_ker = E_ker.cardinality()
        if C_ker%l != 0 :   
            print('Erreur Pas de l torsion : r = 2(l-1)')
            r = 2*(l-1)
            q_ker = q_ker*q_ker
            Fker.<xk>, emb = Fq.extension(r, map = True)
            E_ker = EllipticCurve(Fker, [emb(E.a4()),emb(E.a6())]) 
            C_ker = E_ker.cardinality()
    print('C_ker ok :', C_ker)
    print('taille extension :',r)

    #On calcule la liste des entiers n1 possibles tels que E_ker soit le groupe (Z/n1 Z) x (Z/n2 Z) avec n1|n2. 
    #Ils vérifient n1^2 | C_ker et q_ker = 1 mod n1
    print('début calcul des n1 possibles')
    carrés = facteurs_carrés(C_ker,q_ker)
    k = len(carrés)
    print('Nb de n1 à tester :',k)
    print('Fin calculs noyau extension')
    return q_ker, c, C_ker, carrés


def facteurs_carrés(n,qr):

    #On cherche la liste de tous les entiers n1 tels que n1^2 divise n et qr = 1 mod n1
    nmax = gcd(n,qr-1)
    carrés = []
    n1 = 2
    while n1 <= int(sqrt(nmax)):
        if (nmax%n1 == 0 and n%(n1^2) == 0):
            carrés.append(n1)
        n1 = n1 + 1
    if n%(nmax^2) == 0:
        carrés.append(nmax)
    return carrés

def generateur_noyau(E,l,L,q,kq,f,fm,dk,tracef,q_ker, c, C_ker, carrés):
    
    #On calcule un générateur du noyau de la l-isogénie définie par L.
    
    print('début test générateur')
    Fker.<xk> = GF(q_ker) 
    E_ker = EllipticCurve(Fker, [E.a4(),E.a6()]) 
    k = len(carrés)
    test = False
    essais = 0
    while test == False :
        essais = essais + 1
        P = E_ker.random_point()
        K1 = (C_ker/l)*P
        if (K1 != E_ker(0)) :
            test = (eval_fq(K1,kq,E_ker) == (-c)*K1)
        elif P != E_ker(0):  #On suppose P d'ordre n2. 
            for i in [0 .. k-1] :
                n1 = carrés[i]
                n2 = C_ker/n1 
                if n2%l == 0 and n2*P == E_ker(0):
                    K1 = (n2/l)*P    
                    if (K1 != E_ker(0)) :
                        if (eval_fq(K1,kq,E_ker) == (-c)*K1):
                            print('Nb essais generateur du noyau :')
                            print(essais)
                            K1.set_order(l)    #On evite à sage de recalculer l'ordre du point
                            return E_ker, K1
            test = False
        else :
            test = False
            
    print('Nb essais generateur du noyau :')
    print(essais)
    K1.set_order(l) #On evite à sage de recalculer l'ordre du point
    return E_ker, K1

def phi_from_ker(K,l,E):

    #On calcule une isogénie de degré l et noyau engendré par un point K. On calcule une equation normalisée. 
    
    if l >= 100:
        phi = EllipticCurveHom_velusqrt(E, K) #SquareVélu n'est pas implémenté pour l < 10, et est plus lent pour l < 100. 
    else:
        phi = EllipticCurveIsogeny(E, K) #Applique la formule de vélu. Ne renvoie pas le même type que square vélu, mais compatible.
    
    #normalisation :
    u = phi.scaling_factor() #Calcule l'action sur l'invariant différentiel de la forme explicite calculée
    E1 = phi.codomain()
    isom = WeierstrassIsomorphism(E1, (u^(-1), 0, 0, 0))  
    phi_normal = isom * phi
    return phi_normal

def sqrtvelu(E,q,kq,E_eval,K,O,L):

    q = p^kq
    début = time.time()
    dK = K.discriminant()
    f = O.conductor()
    D = f^2*dK
    tracef = E.trace_of_frobenius()
    Dm = tracef^2 - 4*q
    fm = isqrt(Dm/dK)
    print('constantes dans velu :', dK, tracef, Dm, fm)
    print('Basefield :',E.base_ring())

    
    l = int(L.norm())


    if p > 4*l + 4 and 4*l^2 < (-D) :#Cf borne de Shoof, celle de Broker est fausse
        
        Psi_l = classical_modular_polynomial(l)
        j = E.j_invariant()
        phi = phi_from_L_polynome(E,L,l,q,kq,f,fm,dK,tracef,j,None,Psi_l)

    else:
        q_ker, c, C_ker, carrés = card_extension_noyau(E,l,L,f,fm,dK,tracef)
        E_ker, K = generateur_noyau(E,l,L,q,kq,f,fm,dK,tracef,q_ker, c, C_ker, carrés)
        phi = phi_from_ker(K,l,E_ker)
        
    print('temps :')
    print(time.time() - début)
    
    return phi

# Si b ou c n'est pas premier, il faut factoriser :

def facto_L(L,O):

    # On factorise L à condition que N(L) soit factorisable. TODO version prenant une facto de n en entrée. 
    N = L.norm()
    D = O.discriminant()
    f = O.conductor()
    qL = L.quadratic_form()
    
    a = qL[0]
    m = N/a    
    assert m.is_square()
    m = isqrt(m)
    
    N2 = a.gcd(N)
    m = N/N2
    assert m.is_square()
    m = isqrt(m)
    assert N == N2*(m^2)
    
    factoL = []
    expoL = []
    if N2 > 1:
        facto_N2 = N2.factor()
        b = qL[1]
        B = O
        for facteur in facto_N2:
            p = facteur[0]
            exp = facteur[1]
            if kronecker(D,p) == 1 :
                qp = forme_de_norme(p,D)
                bp = qp[1]
                Ip = NumberFieldOrderIdeal(O,qp)
                expoL.append(exp)
                for _ in range(exp):
                    if Mod(b, 2*p) == Mod(bp, 2*p):
                        B = Ip*B
                        conjug = False
                    else:
                        B = (Ip.conjugate())*B
                        conjug = True
                if conjug:
                    factoL.append(Ip.conjugate())
                else:
                    factoL.append(Ip)
            elif kronecker(D,p) == 0 :
                Ip = ideal_de_norme(p,f,D)
                expoL.append(exp)
                factoL.append(Ip)
            else:
                print(p)
                assert exp % 2 == 0
                me = me*p^(exp//2) #p inerte donc rentre dans le facteur d'idéaux principal entier
    #if m > 1:
        #factoL.append(m*O)
        #expoL.append(1)

    return factoL, expoL, m

def sqrtvelu_compose(P,E,q,kq,K,O,L):
    facto, expo, m = facto_L(L,O)
    n = len(facto)
    départ = E
    Q = P
    Fq = GF(q)
    for i in range(n):
        for _ in range(expo[i]):
            phi = sqrtvelu(départ,q,kq,départ,K,O,facto[i])
            Q = phi(Q)
            départ = phi.codomain()
            départ = départ.change_ring(Fq)
            Q = départ(Q)
            assert Q in départ
    Q = m*Q
    return Q, départ

def eval_theta_SqrtVelu(P,E,K,O,b,c):
    cb = c.conjugate()
    q = E.base_ring().cardinality()
    d = E.base_ring().degree()
    P1, E1 = sqrtvelu_compose(P,E,q,d,K,O,b)
    if b.is_principal():
        assert E.j_invariant() == E1.j_invariant()
    else:
        assert E.j_invariant() != E1.j_invariant()
    P2, E2 = sqrtvelu_compose(P1,E1,q,d,K,O,cb)
    P3, E3 = sqrtvelu_compose(P,E,q,d,K,O,cb)
    if cb.is_principal():
        assert E.j_invariant() == E3.j_invariant()
    else:
        assert E.j_invariant() != E3.j_invariant()
    P4, E4 = sqrtvelu_compose(P3,E3,q,d,K,O,b)
    assert E2 == E4
    assert E2.is_isomorphic(E)
    print('E1 :', E1)
    print('E3 :', E3)
    return P2, E2

# Code : 2_Gluing

In [3]:
from sage.rings.finite_rings.integer_mod import square_root_mod_prime
from sage.rings.finite_rings.integer_mod import square_root_mod_prime_power
from sage.rings.number_field.order_ideal import NumberFieldOrderIdeal

def legendre_adapte(E,P,Q):
    
    #Legendre envoyant P sur (0,0) et Q sur (1,0)
    
    assert P in E
    assert Q in E
    lamb = ((P+Q)[0] - P[0])/(Q[0] - P[0])
    return lamb


def eval_gluing(P1,P2,J,C,r,x):

    lift0 = C.lift_x(0,all = True)
    infs = C.points_at_infinity()

    # préimage de P pour la cover (x,y) -> ( 1/x^2 , ry/x^3 )
    a1 = (x^2)*P1[0] - 1        #Vrai si P[0] est différents de 0 ? 
    b1 = P1[1]*(x^3)*(r^(-1))  
    b1 = b1%a1
    D1 = J([a1,b1])

    # préimage de P pour la cover (x,y) -> (x^2,y)
    a2 = x^2 - P2[0]
    b2 = x + P2[1] - x
    D2 = J([a2,b2])

    evalg = D1 + D2 - (J(infs[0]) + J(infs[1])) - (J(lift0[0]) + J(lift0[1]))
    evalg = evalg + 2*J(infs[1])  #TODO Justifier cette étape ? 2*J(infs[1]) = [ 2(inf+) - 2(inf-) ]

    return evalg


def gluing_legendre(E,EE,kernel,eval):

    #TODO evaluation en gluing, avec coordonée de mumford

    Fq = E.base_ring()
    q = Fq.cardinality()
    p = Fq.characteristic()
    d = Fq.degree()
    PFq.<x> = PolynomialRing(Fq)
    
    P1 = kernel[0][0]
    P2 = kernel[0][1]
    Q1 = kernel[1][0]
    Q2 = kernel[1][1]
    
    R1 = eval[0][0]
    R2 = eval[0][1]
    S1 = eval[1][0]
    S2 = eval[1][1]
    
    if P1 != E(0) and P2 != EE(0) and Q1 != E(0) and Q2 != EE(0):
        lamb = legendre_adapte(E,P1,Q1)
        mu = legendre_adapte(EE,P2,Q2)
        if lamb != mu:  #Gluing
            print('Gluing')

            #print('Kernel gluing :', kernel)
            
            #Mise sous forme de Legendre
            
            flamb = x*(x-1)*(x - lamb)
            Elamb = EllipticCurve([0,flamb[2],0,flamb[1],flamb[0]])
            isolamb = E.isomorphism_to(Elamb)
            
            fmu = x*(x-1)*(x - mu)
            Emu = EllipticCurve([0,fmu[2],0,fmu[1],fmu[0]])
            isomu = EE.isomorphism_to(Emu)

            #Definition de l'image

            f6 = (x^2 - 1)*(x^2 - (lamb/mu))*(x^2 - (lamb - 1)/(mu - 1))
            C6 = HyperellipticCurve(f6)
            J6 = C6.jacobian()

            #Definition du domaine

            f1 = (x-1)*(x - mu/lamb)*(x - (mu-1)/(lamb - 1))
            E1 = EllipticCurve([0,f1[2],0,f1[1],f1[0]])
            assert Elamb.is_isomorphic(E1)
            iso1 = Elamb.isomorphism_to(E1)

            f2 = (x-1)*(x - lamb/mu)*(x - (lamb-1)/(mu - 1))
            E2 = EllipticCurve([0,f2[2],0,f2[1],f2[0]])
            assert Emu.is_isomorphic(E2) and f2(x^2) == f6
            iso2 = Emu.isomorphism_to(E2)

            #Besoin d'une extension de degré 2 pour les calculs intermédiaires

            R = - (mu/lamb)*( (mu - 1)/(lamb - 1) )
            if kronecker(R,p) == -1:
                Fq2.<r> = GF(q^2, modulus = x^2 - R)  #TODO tester si q n'est pas premier
                C = C6(Fq2)
                J = J6(Fq2)
            else:
                r = square_root_mod_prime_power(R,p,d)
                C = C6
                J = J6

            assert R*f2(x^2) == (x^6)*f1(1/x^2)

            # Définition des points à évaluer (R1, R2), (S1, S2)
            
            R1 = iso1(isolamb(R1))
            S1 = iso1(isolamb(S1))
            R2 = iso2(isomu(R2))
            S2 = iso2(isomu(S2))

            #print('isos :', iso1, iso2)

            #print('R :', R1, R2)
            #print('S :', S1, S2)
            
            return (J,C), [eval_gluing(R1,R2,J,C,r,x), eval_gluing(S1,S2,J,C,r,x)]   
            
        else:          #(1,1)_diamant avec id
            print( '(1,1)-diamant' )
            assert E.is_isomorphic(EE)
            if EE != E:
                isom = EE.isomorphism_to(E)
                return (E,E), [[R1 - isom(R2), R1 + isom(R2)], [S1 - isom(S2), S1 + isom(S2)]]
            else:
                return (E,E), [[R1 - R2, R1 + R2], [S1 - S2, S1 + S2]]
            
    else:    #isogenies diagonales
        print('Isogénies Diag')
        if P1 == E(0):
            assert Q1 != E(0) and P2 != EE(0) and (Q2 == EE(0) or Q2 == P2)
            phi1 = E.isogeny(Q1)
            phi2 = EE.isogeny(P2)
        else:
            assert Q1 == E(0) and (P2 == EE(0) or P2 == Q2) and Q2 != EE(0)
            phi1 = E.isogeny(P1)
            phi2 = EE.isogeny(Q2)
            
        return (phi1.codomain(),phi2.codomain()), [[phi1(R1),phi2(R2)],[phi1(S1),phi2(S2)]]

def first_steps_2gluing(E,D,f,Diam):

    # E courbe elliptique ordinaire d'anneau d'endomorphisme de discriminant D.
    # Diam décrit un diamant d'isogénie formé à partir de deux idéaux équivalent 
    
    q = E.base_ring().cardinality()
    Fq = GF(q)
    K.<rK> = QuadraticField(D)
    dK = K.discriminant()
    K.<rK> = QuadraticField(dK)
    wK = (dK + rK)/2
    O = K.order([1,f*wK])
    
    T = Diam[2] # T est une puissance de 2
    N = Diam[0]
    b = Diam[3]
    c = Diam[4]

    assert b.gens()[0] in O 
    assert b.gens()[1] in O 
    assert c.gens()[0] in O 
    assert c.gens()[1] in O 
    assert b.is_equivalent(c)
    
    (P,Q) = E.torsion_basis(T)                  # Dans une version final, on veut éviter ce calcul, le remplacer par la 2 torsion
    OP, E2 = eval_theta_SqrtVelu(P,E,K,O,b,c)   # Peut être lent si besoin de calculer dans une grande extension, dépent des normes de b, c
    OQ, Etest = eval_theta_SqrtVelu(Q,E,K,O,b,c)
    assert Etest == E2
    assert E2.is_isomorphic(E)
    isom = E2.isomorphism_to(E)
    OQ = isom(OQ)
    OP = isom(OP)

    pas = 1
    kernel_T = [[N*P, OP],[N*Q,OQ]]
    domain = (E,E)
    g = 1
    while g == 1:
        kernel_2 = [[(T/(2^pas))*kernel_T[0][0], (T/(2^pas))*kernel_T[0][1]],[(T/(2^pas))*kernel_T[1][0],(T/(2^pas))*kernel_T[1][1]]]
        print('kernel suivant :')
        print(kernel_2)
        print('--------')
        print('Step :')
        Glu = gluing_legendre(domain[0],domain[1],kernel_2,kernel_T)
        Test = gluing_legendre(domain[0],domain[1],kernel_2,kernel_2)
        domain = Glu[0]
        kernel_T = Glu[1]
        print(domain[0])
        print(domain[1])
        print('kernel restant :')
        print(kernel_T)
        domain_test = Test[0]
        eval_test = Test[1]
        pas = pas + 1
        g = domain[1].genus()
        if g == 2:
            assert eval_test[0] == (domain_test[0])(0)
            assert eval_test[1] == (domain_test[0])(0)
        elif g == 1 :
            assert eval_test[0][0] == eval_test[1][0] and eval_test[0][0] == (domain_test[0])(0), (eval_test[0][0], eval_test[0][1])
            assert eval_test[0][1] == eval_test[1][1] and eval_test[0][1] == (domain_test[1])(0), (eval_test[1][0], eval_test[1][1])

    return domain, kernel_T


In [4]:
#Exemple de courbe 1

p = 4295049217
Fp = GF(p)
E = EllipticCurve(Fp, [2857265638,3935716393])
D = -39
f = 1
K.<rK> = QuadraticField(D)
dK = K.discriminant()
K.<rK> = QuadraticField(dK)
wK = (dK + rK)/2
O = K.order([1,f*wK])
b = NumberFieldOrderIdeal(O, [9/2*rK + 1/2, 5*rK])
c = NumberFieldOrderIdeal(O, [107/2*rK + 1/2, 59*rK])
Diam = [b.norm(), c.norm(), b.norm() + c.norm(), b, c]
Diam

[5,
 59,
 64,
 Ideal (9/2*rK + 1/2, 5*rK) of Maximal Order generated by 1/2*rK + 1/2 in Number Field in rK with defining polynomial x^2 + 39 with rK = 6.244997998398398?*I,
 Ideal (107/2*rK + 1/2, 59*rK) of Maximal Order generated by 1/2*rK + 1/2 in Number Field in rK with defining polynomial x^2 + 39 with rK = 6.244997998398398?*I]

In [5]:
domain, kernel_T = first_steps_2gluing(E,D,f,Diam)

constantes dans velu : -39 81922 -10468982784 16384
Basefield : Finite Field of size 4295049217
début calculs extensions noyaux
f: 1 fm : 16384
nb_iso: 2
debut calcul C_ker
C_ker ok : 340308329464935390385026756410710425600
taille extension : 4
début calcul des n1 possibles
Nb de n1 à tester : 17
Fin calculs noyau extension
début test générateur
Nb essais generateur du noyau :
1
temps :
0.17467999458312988
constantes dans velu : -39 81922 -10468982784 16384
Basefield : Finite Field of size 4295049217
début calculs extensions noyaux
f: 1 fm : 16384
nb_iso: 2
debut calcul C_ker
C_ker ok : 2270263143158170335675002865527674960242208345947224258126995847217424701620369471557397584792877552369712175331786692225795218166525851389744982476815474921442086955674308616808896007341376879879911169626364801164551331655531697262884999240162642081090824937874479479538828967936
taille extension : 29
début calcul des n1 possibles
Nb de n1 à tester : 15
Fin calculs noyau extension
début test générateur


In [6]:
domain[1].absolute_igusa_invariants_kohel() #Pour vérifier le résultat : (2863476614, 3754930845, 4078195756)

(2863476614, 3754930845, 4078195756)

In [7]:
kernel_T

[(x^2 + 1219759741*x + 3075289475, 0 : 0),
 (x^2 + 632705604*x + 632705603, 0 : 0)]

In [8]:
J = domain[0]
J

Jacobian of Hyperelliptic Curve over Finite Field of size 4295049217 defined by y^2 = x^6 + 991205405*x^4 + 3796992014*x^2 + 3801901014

In [9]:
C = domain[1]
C

Hyperelliptic Curve over Finite Field of size 4295049217 defined by y^2 = x^6 + 991205405*x^4 + 3796992014*x^2 + 3801901014

In [10]:
K1 = kernel_T[0]
K2 = kernel_T[1]
K1, K2, 32*K1, 32*K2

((x^2 + 1219759741*x + 3075289475, 0 : 0),
 (x^2 + 632705604*x + 632705603, 0 : 0),
 (1, 0 : 1),
 (1, 0 : 1))

In [11]:
#Exemple 2

p = 17179754497
Fp = GF(p)
E = EllipticCurve(Fp, [12383769515,16144520740])
D = -23
f = 1
K.<rK> = QuadraticField(D)
dK = K.discriminant()
K.<rK> = QuadraticField(dK)
wK = (dK + rK)/2
O = K.order([1,f*wK])
b = NumberFieldOrderIdeal(O, [41/2*rK + 1/2, 27*rK])
c = NumberFieldOrderIdeal(O, [169/2*rK + 1/2, 101*rK])
Diam = [b.norm(), c.norm(), b.norm() + c.norm(), b, c]
Diam


[27,
 101,
 128,
 Ideal (41/2*rK + 1/2, 27*rK) of Maximal Order generated by 1/2*rK + 1/2 in Number Field in rK with defining polynomial x^2 + 23 with rK = 4.795831523312720?*I,
 Ideal (169/2*rK + 1/2, 101*rK) of Maximal Order generated by 1/2*rK + 1/2 in Number Field in rK with defining polynomial x^2 + 23 with rK = 4.795831523312720?*I]

In [12]:
domain, kernel_T = first_steps_2gluing(E,D,f,Diam)

constantes dans velu : -23 -114686 -55566139392 49152
Basefield : Finite Field of size 17179754497
début calculs extensions noyaux
f: 1 fm : 49152
nb_iso: 4
debut calcul C_ker
C_ker ok : 295143964598398353408
taille extension : 2
début calcul des n1 possibles
Nb de n1 à tester : 16
Fin calculs noyau extension
début test générateur
Nb essais generateur du noyau :
1
temps :
0.03250265121459961
constantes dans velu : -23 -114686 -55566139392 49152
Basefield : Finite Field of size 17179754497
début calculs extensions noyaux
f: 1 fm : 49152
nb_iso: 4
debut calcul C_ker
C_ker ok : 295143964598398353408
taille extension : 2
début calcul des n1 possibles
Nb de n1 à tester : 16
Fin calculs noyau extension
début test générateur
Nb essais generateur du noyau :
2
temps :
0.024663686752319336
constantes dans velu : -23 -114686 -55566139392 49152
Basefield : Finite Field of size 17179754497
début calculs extensions noyaux
f: 1 fm : 49152
nb_iso: 4
debut calcul C_ker
C_ker ok : 295143964598398353408


In [15]:
#Exemple 3

p = 4295049217
Fp = GF(p)
E = EllipticCurve(Fp, [1598860649,2757655473])
D = -156
f = 2
K.<rK> = QuadraticField(D)
dK = K.discriminant()
K.<rK> = QuadraticField(dK)
wK = (dK + rK)/2
O = K.order([1,f*wK])
b = NumberFieldOrderIdeal(O, [13*rK + 3, 25*rK])
c = NumberFieldOrderIdeal(O, [109*rK + 1, 181*rK])
Diam = [b.norm(), c.norm(), b.norm() + c.norm(), b, c]
Diam

[75,
 181,
 256,
 Ideal (13*rK + 3, 25*rK) of Order of conductor 2 generated by rK in Number Field in rK with defining polynomial x^2 + 39 with rK = 6.244997998398398?*I,
 Ideal (109*rK + 1, 181*rK) of Order of conductor 2 generated by rK in Number Field in rK with defining polynomial x^2 + 39 with rK = 6.244997998398398?*I]

In [27]:
domain, kernel_T = first_steps_2gluing(E,D,f,Diam)  #Erreur calcul de l'endomorphisme theta. Source ?

constantes dans velu : -39 81922 -10468982784 16384
Basefield : Finite Field of size 4295049217
Calcul du polynome du noyau
Recherche de j_invariants
j_invariants trouvés
Choix j_invariant, application BMSS
Verif choix j_invariant
bon j_invariant
Calcul isogénie par Vélu
temps :
0.00831913948059082
constantes dans velu : -39 81922 -10468982784 16384
Basefield : Finite Field of size 4295049217
Calcul du polynome du noyau
Recherche de j_invariants
j_invariants trouvés
Choix j_invariant, application BMSS
Verif choix j_invariant
bon j_invariant
Calcul isogénie par Vélu
temps :
0.008857965469360352
constantes dans velu : -39 81922 -10468982784 16384
Basefield : Finite Field of size 4295049217
Calcul du polynome du noyau
Recherche de j_invariants
j_invariants trouvés
Choix j_invariant, application BMSS
Verif choix j_invariant
bon j_invariant
Calcul isogénie par Vélu
temps :
0.010945558547973633
constantes dans velu : -39 81922 -10468982784 16384
Basefield : Finite Field of size 4295049217
dé

KeyboardInterrupt: 

Exception ignored in: 'sage.symbolic.expression.py_is_real'
Traceback (most recent call last):
  File "/var/tmp/sage-10.9/local/lib/python3.14/site-packages/sage/categories/sets_cat.py", line 1906, in is_finite
    def is_finite(self):
  File "cysignals/signals.pyx", line 357, in cysignals.signals.python_check_interrupt
KeyboardInterrupt: 


KeyboardInterrupt: 

In [16]:
b = NumberFieldOrderIdeal(O, [4*rK + 1, 5*rK])
c = NumberFieldOrderIdeal(O, [48*rK + 1, 59*rK])
Diam = [b.norm(), c.norm(), b.norm() + c.norm(), b, c]
Diam, b.is_principal()

([5,
  59,
  64,
  Ideal (4*rK + 1, 5*rK) of Order of conductor 2 generated by rK in Number Field in rK with defining polynomial x^2 + 39 with rK = 6.244997998398398?*I,
  Ideal (48*rK + 1, 59*rK) of Order of conductor 2 generated by rK in Number Field in rK with defining polynomial x^2 + 39 with rK = 6.244997998398398?*I],
 False)

In [29]:
domain, kernel_T = first_steps_2gluing(E,D,f,Diam)  

constantes dans velu : -39 81922 -10468982784 16384
Basefield : Finite Field of size 4295049217
Calcul du polynome du noyau
Recherche de j_invariants
j_invariants trouvés
Choix j_invariant, application BMSS
Verif choix j_invariant
bon j_invariant
Calcul isogénie par Vélu
temps :
0.015899658203125
constantes dans velu : -39 81922 -10468982784 16384
Basefield : Finite Field of size 4295049217
début calculs extensions noyaux
f: 2 fm : 16384
nb_iso: 2
debut calcul C_ker
C_ker ok : 51540947391824150158228119191412745016323253903977693083819547955088948109728159337360334417240866634614682166284183765804798675981072751220907595885028106251930478314700689133847594277050739721617015049679891630046474059255800890360251677293454235561873855799579173865931232382491748166576413829163113425137284642802740092784511721076006721792005357934548342300687944442877238845161678960600251462557473066166091938501342926020025377307786476234138922342407679868625538096682761496256086827830449741994759863162452914

/tmp/ipykernel_11334/3393114731.py:337: RuntimeWarning: cypari2 leaked 3077784 bytes on the PARI stack
  if n2%l == Integer(0) and n2*P == E_ker(Integer(0)):


KeyboardInterrupt: 

# Richelot Isogenies

In [31]:
# depuis two_isogenies/thetha_SageMath/richelot_isogenies/richelot_isogenies.py